# 1. Import Library

In [1]:
# ==========================================
# CELL 1: IMPORT LIBRARY
# ==========================================
import pandas as pd
import numpy as np
import glob
import os

# Visualisasi
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from imblearn.over_sampling import SMOTE
import xgboost as xgb

# Set tampilan pandas agar kolom tidak terpotong
pd.set_option('display.max_columns', None)
print("✅ Library berhasil dimuat!")

✅ Library berhasil dimuat!


# 2. Load Dataset

In [2]:
# ==========================================
# CELL 2: LOAD DATASET
# ==========================================
# Pastikan path ini menunjuk ke folder tempat CSV-mu berada
dataset_path = '../../EOI-skillselect-scrapping/DATASET/**/*.csv' 
all_files = glob.glob(dataset_path, recursive=True)

df_list = []
for file in all_files:
    df_temp = pd.read_csv(file)
    df_list.append(df_temp)

# Gabungkan seluruh data
df_raw = pd.concat(df_list, ignore_index=True)

print(f"Total baris data awal: {df_raw.shape[0]}")
df_raw.head() # Menampilkan 5 baris pertama

Total baris data awal: 2148242


,As At Month,Visa Type,Occupation,EOI Status,Points,Count EOIs,Nominated State
0,03/2024,190SAS Skilled Australian Sponsored,131112 Sales and Marketing Manager,SUBMITTED,60,<20,ACT
1,03/2024,190SAS Skilled Australian Sponsored,131112 Sales and Marketing Manager,SUBMITTED,65,<20,ACT
2,03/2024,190SAS Skilled Australian Sponsored,131112 Sales and Marketing Manager,SUBMITTED,70,<20,ACT
3,03/2024,190SAS Skilled Australian Sponsored,131112 Sales and Marketing Manager,SUBMITTED,75,<20,ACT
4,03/2024,190SAS Skilled Australian Sponsored,131112 Sales and Marketing Manager,SUBMITTED,80,<20,ACT


# 3. Data Preprocessing

In [3]:
# ==========================================
# CELL 3: DATA PRE-PROCESSING & EXPANSION
# ==========================================
df = df_raw.copy()

# 1. Fokus pada data yang relevan (Abaikan status CLOSED atau HOLD)
df = df[df['EOI Status'].isin(['SUBMITTED', 'INVITED', 'LODGED'])].copy()

# 2. Tangani nilai '<20' di kolom Count EOIs (Kita ubah menjadi 10 sebagai nilai tengah)
df['Count EOIs'] = df['Count EOIs'].astype(str).str.replace('<20', '10')
df['Count EOIs'] = pd.to_numeric(df['Count EOIs'], errors='coerce').fillna(0).astype(int)

# 3. Tentukan Target ML (1 = Sukses Diundang, 0 = Masih Mengantri)
df['Target'] = np.where(df['EOI Status'].isin(['INVITED', 'LODGED']), 1, 0)

# 4. DATASET EXPANSION (Meledakkan baris agregat menjadi baris individu)
# Jika ada 10 orang dengan status sama, baris akan diduplikasi menjadi 10 baris terpisah
df_expanded = df.loc[df.index.repeat(df['Count EOIs'])].reset_index(drop=True)

# 5. Ekstraksi Fitur Waktu
df_expanded['As At Month'] = pd.to_datetime(df_expanded['As At Month'], format='%m/%Y')
df_expanded['Month'] = df_expanded['As At Month'].dt.month
df_expanded['Year'] = df_expanded['As At Month'].dt.year

# 6. Buang kolom yang sudah tidak diperlukan
df_ml = df_expanded.drop(columns=['Count EOIs', 'EOI Status', 'As At Month'])

print(f"Total baris setelah ekspansi (individu): {df_ml.shape[0]}")
print("\nDistribusi Target (0 = Submitted, 1 = Invited):")
print(df_ml['Target'].value_counts())
df_ml.head()

Total baris setelah ekspansi (individu): 36279788

Distribusi Target (0 = Submitted, 1 = Invited):
Target
0    32936011
1     3343777
Name: count, dtype: int64


,Visa Type,Occupation,Points,Nominated State,Target,Month,Year
0,190SAS Skilled Australian Sponsored,131112 Sales and Marketing Manager,60,ACT,0,3,2024
1,190SAS Skilled Australian Sponsored,131112 Sales and Marketing Manager,60,ACT,0,3,2024
2,190SAS Skilled Australian Sponsored,131112 Sales and Marketing Manager,60,ACT,0,3,2024
3,190SAS Skilled Australian Sponsored,131112 Sales and Marketing Manager,60,ACT,0,3,2024
4,190SAS Skilled Australian Sponsored,131112 Sales and Marketing Manager,60,ACT,0,3,2024


# 4. Feature Encoding & Splitting

In [ ]:
# ==========================================
# CELL 4: FEATURE ENCODING & SPLITTING
# ==========================================
# Simpan encoder ke dalam dictionary agar nanti bisa dipakai untuk prediksi data baru
encoders = {}
categorical_cols = ['Visa Type', 'Occupation', 'Nominated State']

for col in categorical_cols:
    le = LabelEncoder()
    df_ml[col] = le.fit_transform(df_ml[col].astype(str))
    encoders[col] = le

# Pisahkan Fitur (X) dan Target (y)
X = df_ml.drop(columns=['Target'])
y = df_ml['Target']

# Bagi data: 80% untuk latihan (Train), 20% untuk ujian (Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Data Latih (Train): {X_train.shape[0]} baris")
print(f"Data Uji (Test): {X_test.shape[0]} baris")